# Gemini

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "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"
model_name = "google/gemini-2.5-flash-lite"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_path = kagglehub.dataset_download("nikhileswarkomati/suicide-watch")
data = pd.read_csv(dataset_path + "/Suicide_Detection.csv").sample(frac=0.3, random_state=0)
data["label"] = data["class"].astype("category").cat.codes
texts = data["text"].astype(str).tolist()
labels = data["label"].tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2,
                reasoning_effort = "none"
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

Using Colab cache for faster access to the 'suicide-watch' dataset.


100%|██████████| 63/63 [10:24<00:00,  9.92s/it]

Inference time: 624.9376357199999
Accuracy: 0.9200
False positive: 75
False negative: 85
Precision: 0.9256689791873142
Recall: 0.9165848871442591
F1: 0.9211045364891519
ROC-AUC: 0.9200661438779401
formatted mistake:  0.0
0
2000


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "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"
model_name = "google/gemini-2.5-flash-lite"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
dataset_path = kagglehub.dataset_download("rvarun11/suicidal-ideation-reddit-dataset")
data = pd.read_csv(dataset_path + "/suicidal_ideation_reddit_annotated.csv")
data["text"] = data["usertext"].astype(str)
texts = data["text"].tolist()
labels = data["label"].astype(int).tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2,
                reasoning_effort = "none"
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

Using Colab cache for faster access to the 'suicidal-ideation-reddit-dataset' dataset.


 98%|█████████▊| 62/63 [10:36<00:10, 10.01s/it]


API error: 'NoneType' object is not subscriptable


100%|██████████| 63/63 [10:40<00:00, 10.17s/it]

Inference time: 640.695455266
Accuracy: 0.8819
False positive: 165
False negative: 71
Precision: 0.8580034423407917
Recall: 0.9335205992509363
F1: 0.8941704035874439
ROC-AUC: 0.8781459064997968
formatted mistake:  0.0005002501250625312
1
1999


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "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"
model_name = "google/gemini-2.5-flash-lite"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
mapping = {
    "Not Suicide post": 0,
    "Potential Suicide post ": 1
}
dataset_path = kagglehub.dataset_download("aunanya875/suicidal-tweet-detection-dataset")
data = pd.read_csv(dataset_path + "/Suicide_Ideation_Dataset(Twitter-based).csv")
data["text"] = data["Tweet"]
data["Suicide"] = data["Suicide"].map(mapping)
data = data.dropna(subset=["Tweet", "Suicide"])
data["Suicide"] = data["Suicide"].astype(int)
texts = data["text"].tolist()
labels = data["Suicide"].tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2,
                reasoning_effort = "none"
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

100%|██████████| 88.3k/88.3k [00:00<00:00, 54.8MB/s]

Extracting files...



100%|██████████| 56/56 [09:09<00:00,  9.81s/it]

Inference time: 549.5976273269998
Accuracy: 0.8303
False positive: 20
False negative: 283
Precision: 0.9494949494949495
Recall: 0.5705614567526556
F1: 0.7127962085308057
ROC-AUC: 0.7763997337049245
formatted mistake:  0.0
0
1785


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "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"
model_name = "google/gemini-2.5-flash-lite"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
mapping = {
    "non-suicide": 0,
    "suicide": 1
}
dataset_path = kagglehub.dataset_download("neelakshigupta/suicide-detection")
data = pd.read_csv(dataset_path + "/file1.csv")
data["text"] = data["text"]
data["Suicide"] = data["class"].map(mapping)
data = data.dropna(subset = ["text", "class"])
data["Suicide"] = data["Suicide"].astype(int)
texts = data["text"].tolist()
labels = data["Suicide"].tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2,
                reasoning_effort="none"
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

100%|██████████| 1.45M/1.45M [00:00<00:00, 122MB/s]

Extracting files...



100%|██████████| 63/63 [10:41<00:00, 10.18s/it]

Inference time: 641.4473528880008
Accuracy: 0.8745
False positive: 218
False negative: 33
Precision: 0.8135158254918734
Recall: 0.9664634146341463
F1: 0.8834184858337204
ROC-AUC: 0.8759482427501439
formatted mistake:  0.0
0
2000


# GPT-5

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "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"
model_name = "openai/gpt-4.1-mini"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_path = kagglehub.dataset_download("nikhileswarkomati/suicide-watch")
data = pd.read_csv(dataset_path + "/Suicide_Detection.csv").sample(frac=0.3, random_state=0)
data["label"] = data["class"].astype("category").cat.codes
texts = data["text"].astype(str).tolist()
labels = data["label"].tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_completion_tokens = 2,
                reasoning_effort = "none"
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

Using Colab cache for faster access to the 'suicide-watch' dataset.


100%|██████████| 63/63 [20:34<00:00, 19.60s/it]

Inference time: 1234.8417664160006
Accuracy: 0.9295
False positive: 98
False negative: 43
Precision: 0.9083255378858747
Recall: 0.9575936883629191
F1: 0.9323091694671147
ROC-AUC: 0.9291011038163481
formatted mistake:  0.0
0
2000


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "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"
model_name = "openai/gpt-4.1-mini"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
dataset_path = kagglehub.dataset_download("rvarun11/suicidal-ideation-reddit-dataset")
data = pd.read_csv(dataset_path + "/suicidal_ideation_reddit_annotated.csv")
data["text"] = data["usertext"].astype(str)
texts = data["text"].tolist()
labels = data["label"].astype(int).tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2,
                reasoning_effort = "none"
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

Using Colab cache for faster access to the 'suicidal-ideation-reddit-dataset' dataset.


100%|██████████| 63/63 [21:20<00:00, 20.33s/it]

Inference time: 1280.8154362739988
Accuracy: 0.8825
False positive: 173
False negative: 62
Precision: 0.8507333908541846
Recall: 0.9408396946564885
F1: 0.8935206162211147
ROC-AUC: 0.8795585027904291
formatted mistake:  0.0
0
2000


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "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"
model_name = "openai/gpt-4.1-mini"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
mapping = {
    "Not Suicide post": 0,
    "Potential Suicide post ": 1
}
dataset_path = kagglehub.dataset_download("aunanya875/suicidal-tweet-detection-dataset")
data = pd.read_csv(dataset_path + "/Suicide_Ideation_Dataset(Twitter-based).csv")
data["text"] = data["Tweet"]
data["Suicide"] = data["Suicide"].map(mapping)
data = data.dropna(subset=["Tweet", "Suicide"])
data["Suicide"] = data["Suicide"].astype(int)
texts = data["text"].tolist()
labels = data["Suicide"].tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2,
                reasoning_effort = "none"
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

Using Colab cache for faster access to the 'suicidal-tweet-detection-dataset' dataset.


100%|██████████| 56/56 [18:47<00:00, 20.14s/it]

Inference time: 1127.583390288999
Accuracy: 0.8398
False positive: 23
False negative: 263
Precision: 0.9451073985680191
Recall: 0.6009104704097117
F1: 0.7346938775510204
ROC-AUC: 0.7902420913327421
formatted mistake:  0.0
0
1785


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from openai import OpenAI
import numpy as np
key = "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"
model_name = "openai/gpt-4.1-mini"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import kagglehub
mapping = {
    "non-suicide": 0,
    "suicide": 1
}
dataset_path = kagglehub.dataset_download("neelakshigupta/suicide-detection")
data = pd.read_csv(dataset_path + "/file1.csv")
data["text"] = data["text"]
data["Suicide"] = data["class"].map(mapping)
data = data.dropna(subset = ["text", "class"])
data["Suicide"] = data["Suicide"].astype(int)
texts = data["text"].tolist()
labels = data["Suicide"].tolist()
indices = torch.randperm(len(texts))[:2000].tolist()
texts  = [texts[i] for i in indices]
labels = [labels[i] for i in indices]
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts = [b[0] for b in batch]
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return texts, labels

dataset = TextDataset(texts, labels)
eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

cache = {}
formatted_mistake=0
def classify(batch):
    global formatted_mistake
    preds = []
    indices = []
    for i, text in enumerate(batch):
        parsed = None
        prompt = (
            "Output only 0 or 1.\n"
            "0 = no suicidal indicators\n"
            "1 = suicidal indicators\n"
            f"Text: {text}\n"
            "Output:"
        )
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2,
                reasoning_effort="none"
            )

            content = r.choices[0].message.content

            if content is None:
                print("\nEMPTY RESPONSE")
                print("finish_reason:", r.choices[0].finish_reason)
                formatted_mistake += 1
                parsed = 0
                continue

            else:
                out = content.strip()

                if out == "1":
                    parsed = 1
                elif out == "0":
                    parsed = 0
                else:
                    print("\nUnexpected output:", repr(out))
                    formatted_mistake += 1
                    continue

        except Exception as e:
            print("\nAPI error:", e)
            formatted_mistake += 1
            parsed = 0
            continue

        cache[text] = parsed
        preds.append(parsed)
        indices.append(i)

    return preds, indices

total = 0
correct = 0

false_positive = 0
false_negative = 0
import time
preds = []
final_labels = []
t0 = time.perf_counter()
for batch_text, batch_labels in tqdm(eval_loader):
    batch_preds, batch_indices = classify(batch_text)
    batch_preds = torch.tensor(batch_preds, dtype=torch.long).to(device)
    batch_labels = batch_labels[batch_indices].to(device)
    preds.append(batch_preds.cpu().numpy())
    final_labels.append(batch_labels.cpu().numpy())
    total += len(batch_labels)
    correct += (batch_preds == batch_labels).sum().item()
    false_positive += ((batch_preds == 1) & (batch_labels == 0)).sum().item()
    false_negative += ((batch_preds == 0) & (batch_labels == 1)).sum().item()
t1 = time.perf_counter()
preds = np.concatenate(preds)
final_labels = np.concatenate(final_labels)
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
print(f"Inference time: {t1-t0}")
print(f"Accuracy: {correct/total:.4f}")
print(f"False positive: {false_positive}")
print(f"False negative: {false_negative}")
print(f"Precision: {precision_score(final_labels, preds)}")
print(f"Recall: {recall_score(final_labels, preds)}")
print(f"F1: {f1_score(final_labels, preds)}")
print(f"ROC-AUC: {roc_auc_score(final_labels, preds)}")
import time
print("formatted mistake: ", formatted_mistake/total)
print(formatted_mistake)
print(total)

Using Colab cache for faster access to the 'suicide-detection' dataset.


100%|██████████| 63/63 [21:56<00:00, 20.90s/it]

Inference time: 1316.779013768999
Accuracy: 0.8650
False positive: 248
False negative: 22
Precision: 0.7943615257048093
Recall: 0.9775510204081632
F1: 0.8764867337602927
ROC-AUC: 0.8672068827531012
formatted mistake:  0.0
0
2000
